In [1]:
# Cell 1 — Imports & Config
import sys; sys.path.insert(0, '..')
import numpy as np
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

from src.data import load_train, load_test, CLASSES
from src.features import build_features, build_sequences
from src.metrics import compute_map, print_results
from src.submission import save_submission

# ── Config ──────────────────────────────────────────────────────────
N_STEPS          = 64
N_FOLDS          = 5
N_EPOCHS         = 150
PATIENCE         = 25    # was 15 — model stopped at epoch 17, too early
BATCH            = 64
LR               = 1e-3
WD               = 1e-4
TEMP             = 1.0   # was 1.5 — don't soften an already underconfident model
MIXUP_TARGET     = 200   # oversample minority classes to this count per fold
MIXUP_ALPHA      = 0.4   # Beta(0.4, 0.4) — moderate mixing
SEED             = 42

# Device: CUDA > Apple MPS > CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print('All imports OK')
print(f'Device: {DEVICE}')
print(f'Classes: {CLASSES}')

All imports OK
Device: mps
Classes: ['Clutter', 'Cormorants', 'Pigeons', 'Ducks', 'Geese', 'Gulls', 'Birds of Prey', 'Waders', 'Songbirds']


In [ ]:
# Cell 2 — Load Data & Extract Features
train = load_train()
test  = load_test()
print(f'Train: {train.shape}, Test: {test.shape}')

# Tabular features (for GBMs)
print('\nExtracting tabular features...')
X_df      = build_features(train)
X_test_df = build_features(test)
X         = X_df.values
X_test    = X_test_df.values
feature_names = list(X_df.columns)
print(f'Tabular: train={X.shape}, test={X_test.shape}')

# Sequence features (for CNN)
print('\nExtracting sequence features...')
seqs_train = build_sequences(train, n_steps=N_STEPS)
seqs_test  = build_sequences(test,  n_steps=N_STEPS)
print(f'Sequences: train={seqs_train.shape}, test={seqs_test.shape}')

# Encode targets & class weights
le = LabelEncoder().fit(CLASSES)
y  = le.transform(train['bird_group'])
n_classes = len(CLASSES)
class_counts  = np.bincount(y)
class_weights = (1 / class_counts) / (1 / class_counts).sum() * n_classes
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print(f'\nClass distribution: {np.bincount(y)}')
print(f'Class weights: {class_weights.round(3)}')

Train: (2601, 16), Test: (1872, 9)

Extracting tabular features...


Tabular: train=(2601, 73), test=(1872, 73)

Extracting sequence features...


Sequences: train=(2601, 6, 64), test=(1872, 6, 64)

Class distribution: [ 108   84   40   58   83 1503  122  483  120]
Class weights: [0.88  1.131 2.375 1.638 1.145 0.063 0.779 0.197 0.792]


In [3]:
# Cell 3 — BirdDataset & BirdCNN Model

class BirdDataset(Dataset):
    """Wraps (N, 6, 64) sequence array and optional integer labels."""

    def __init__(self, sequences: np.ndarray, labels: np.ndarray = None):
        self.X = torch.from_numpy(sequences).float()
        self.y = torch.from_numpy(labels).long() if labels is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return (self.X[idx],)


class ResConvBlock(nn.Module):
    """Two Conv1d layers with BN+ReLU and a residual skip connection."""

    def __init__(self, in_ch: int, out_ch: int, kernel: int):
        super().__init__()
        pad = kernel // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, padding=pad), nn.BatchNorm1d(out_ch), nn.ReLU(),
            nn.Conv1d(out_ch, out_ch, kernel, padding=pad), nn.BatchNorm1d(out_ch), nn.ReLU(),
        )
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.conv(x) + self.skip(x)


class BirdCNN(nn.Module):
    """3-block residual 1D-CNN: 6-channel x 64-step input -> 9-class output."""

    def __init__(self, n_channels: int = 6, n_classes: int = 9):
        super().__init__()
        self.blocks = nn.Sequential(
            ResConvBlock(n_channels,  64, kernel=7),
            ResConvBlock(64,         128, kernel=5),
            ResConvBlock(128,        256, kernel=3),
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.blocks(x)       # (B, 256, 64)
        x = x.mean(dim=-1)      # global average pooling -> (B, 256)
        return self.head(x)      # (B, n_classes) — raw logits

# Quick forward pass sanity check
_dummy = torch.randn(4, 6, 64)
_out   = BirdCNN()(_dummy)
assert _out.shape == (4, 9)
print('BirdCNN forward pass OK:', _out.shape)

BirdCNN forward pass OK: torch.Size([4, 9])


In [4]:
# Cell 4 — Training Helpers

def mixup_within_class(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    target_count: int = 200,
    alpha: float = 0.4,
    rng: np.random.Generator = None,
) -> tuple:
    """
    Within-class MixUp: generate synthetic sequences for minority classes.

    For each class with fewer than target_count samples in X_tr, randomly pairs
    real samples from that class and blends them with λ ~ Beta(alpha, alpha).
    Same-class blending keeps the synthetic tracks physically plausible.

    Returns X_aug (N+M, 6, 64) and y_aug (N+M,) with synthetic samples appended.
    """
    if rng is None:
        rng = np.random.default_rng(42)

    syn_X, syn_y = [], []
    classes, counts = np.unique(y_tr, return_counts=True)

    for cls, cnt in zip(classes, counts):
        if cnt >= target_count:
            continue
        n_needed = target_count - cnt
        cls_idx  = np.where(y_tr == cls)[0]
        for _ in range(n_needed):
            # replace=True handles the case of a single-sample class
            i, j = rng.choice(cls_idx, 2, replace=True)
            lam   = float(rng.beta(alpha, alpha))
            syn_X.append(lam * X_tr[i] + (1 - lam) * X_tr[j])
            syn_y.append(cls)

    if syn_X:
        X_aug = np.concatenate([X_tr, np.stack(syn_X).astype(np.float32)])
        y_aug = np.concatenate([y_tr, np.array(syn_y, dtype=np.int64)])
        return X_aug, y_aug
    return X_tr, y_tr


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_b)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_proba(model, sequences: np.ndarray) -> np.ndarray:
    """Run model inference, apply temperature scaling, return softmax probs."""
    model.eval()
    ds     = BirdDataset(sequences)
    loader = DataLoader(ds, batch_size=256, shuffle=False)
    logits_list = []
    for (X_b,) in loader:
        logits_list.append(model(X_b.to(DEVICE)).cpu())
    logits = torch.cat(logits_list, dim=0)
    return torch.softmax(logits / TEMP, dim=-1).numpy()

print('Training helpers defined (mixup_within_class, train_one_epoch, predict_proba).')

Training helpers defined (mixup_within_class, train_one_epoch, predict_proba).


In [5]:
# Cell 5 — CNN 5-Fold CV (with within-class MixUp)
print('=== Training 1D-CNN ===\n')

oof_cnn  = np.zeros((len(y), n_classes))
test_cnn = np.zeros((len(seqs_test), n_classes))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(skf.split(seqs_train, y)):
    print(f'--- CNN Fold {fold+1}/{N_FOLDS} ---')
    X_tr, X_va = seqs_train[tr_idx], seqs_train[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Within-class MixUp: generate synthetic sequences for minority classes
    # target_count=200 brings all minority classes up to 200 real+synthetic samples
    rng = np.random.default_rng(SEED + fold)
    X_tr_mx, y_tr_mx = mixup_within_class(
        X_tr, y_tr, target_count=MIXUP_TARGET, alpha=MIXUP_ALPHA, rng=rng
    )
    aug_counts = np.bincount(y_tr_mx, minlength=n_classes)
    print(f'  MixUp: {len(y_tr)} → {len(y_tr_mx)} samples | per-class: {aug_counts}')

    # WeightedRandomSampler on the augmented set
    sample_weights = class_weights_tensor[y_tr_mx].numpy()
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_tr_mx), replacement=True)

    tr_loader = DataLoader(BirdDataset(X_tr_mx, y_tr_mx), batch_size=BATCH, sampler=sampler)
    va_loader = DataLoader(BirdDataset(X_va,    y_va),    batch_size=256,  shuffle=False)

    model     = BirdCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(DEVICE), label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    for epoch in range(N_EPOCHS):
        train_loss = train_one_epoch(model, tr_loader, optimizer, criterion)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in va_loader:
                val_loss += criterion(model(X_b.to(DEVICE)), y_b.to(DEVICE)).item() * len(X_b)
        val_loss /= len(va_idx)
        scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        if (epoch + 1) % 25 == 0:
            print(f'  Epoch {epoch+1:3d}: train={train_loss:.4f}  val={val_loss:.4f}  patience={patience_count}')

        if patience_count >= PATIENCE:
            print(f'  Early stop at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    oof_cnn[va_idx] = predict_proba(model, X_va)
    test_cnn       += predict_proba(model, seqs_test) / N_FOLDS

    fold_map, _ = compute_map(y_va, oof_cnn[va_idx])
    print(f'  Fold {fold+1} CNN OOF mAP: {fold_map:.4f}\n')

cnn_map, cnn_per_class = compute_map(y, oof_cnn)
print_results(cnn_map, cnn_per_class, label='CNN only (OOF)')

=== Training 1D-CNN ===

--- CNN Fold 1/5 ---


  MixUp: 2080 → 2988 samples | per-class: [ 200  200  200  200  200 1202  200  386  200]


  Epoch  25: train=1.0050  val=4.1343  patience=7


  Early stop at epoch 43


  Fold 1 CNN OOF mAP: 0.1884

--- CNN Fold 2/5 ---
  MixUp: 2081 → 2989 samples | per-class: [ 200  200  200  200  200 1202  200  387  200]


...

  Epoch  25: train=1.0750  val=4.6254  patience=24


  Early stop at epoch 26
  Fold 4 CNN OOF mAP: 0.1762

--- CNN Fold 5/5 ---


  MixUp: 2081 → 2989 samples | per-class: [ 200  200  200  200  200 1203  200  386  200]


  Epoch  25: train=1.1371  val=4.0044  patience=24


  Early stop at epoch 26
  Fold 5 CNN OOF mAP: 0.1685


  CNN only (OOF)

  Overall mAP: 0.1523

  Clutter        : 0.2626 <-- weak
  Cormorants     : 0.0717 <-- weak
  Pigeons        : 0.0184 <-- weak
  Ducks          : 0.0664 <-- weak
  Geese          : 0.0869 <-- weak
  Gulls          : 0.5865 <-- weak
  Birds of Prey  : 0.0514 <-- weak
  Waders         : 0.1841 <-- weak
  Songbirds      : 0.0422 <-- weak


In [6]:
# Cell 6 — GBM 5-Fold CV (LGB + XGB + CB)
print('=== Training GBM Ensemble ===\n')

oof_lgb  = np.zeros((len(X), n_classes))
oof_xgb  = np.zeros((len(X), n_classes))
oof_cb   = np.zeros((len(X), n_classes))
test_lgb = np.zeros((len(X_test), n_classes))
test_xgb = np.zeros((len(X_test), n_classes))
test_cb  = np.zeros((len(X_test), n_classes))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    print(f'--- GBM Fold {fold+1}/{N_FOLDS} ---')
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    smote = SMOTE(sampling_strategy='not majority', random_state=SEED + fold)
    X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
    print(f'  SMOTE: {len(y_tr)} -> {len(y_tr_sm)} samples')

    # LightGBM
    lgb_params = dict(
        objective='multiclass', num_class=n_classes, metric='multi_logloss',
        learning_rate=0.05, num_leaves=47, max_depth=7,
        min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, is_unbalance=True,
        verbose=-1, seed=SEED, n_jobs=-1,
    )
    dtrain  = lgb.Dataset(X_tr_sm, label=y_tr_sm, feature_name=feature_names)
    dval    = lgb.Dataset(X_va, label=y_va, reference=dtrain)
    lgb_m   = lgb.train(lgb_params, dtrain, 2000, valid_sets=[dval],
                        callbacks=[lgb.early_stopping(80), lgb.log_evaluation(400)])
    oof_lgb[va_idx]  = lgb_m.predict(X_va)
    test_lgb        += lgb_m.predict(X_test) / N_FOLDS

    # XGBoost
    sw    = class_weights[y_tr_sm]
    xgb_m = xgb.XGBClassifier(
        n_estimators=2000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', early_stopping_rounds=80,
        seed=SEED, n_jobs=-1, verbosity=0,
    )
    xgb_m.fit(X_tr_sm, y_tr_sm, sample_weight=sw,
              eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[va_idx]  = xgb_m.predict_proba(X_va)
    test_xgb        += xgb_m.predict_proba(X_test) / N_FOLDS

    # CatBoost
    cb_m = CatBoostClassifier(
        iterations=2000, learning_rate=0.05, depth=6,
        loss_function='MultiClass', eval_metric='TotalF1',
        class_weights=class_weights.tolist(),
        early_stopping_rounds=80, random_seed=SEED, verbose=400,
    )
    cb_m.fit(X_tr_sm, y_tr_sm, eval_set=(X_va, y_va))
    oof_cb[va_idx]  = cb_m.predict_proba(X_va)
    test_cb        += cb_m.predict_proba(X_test) / N_FOLDS

    fold_map, _ = compute_map(y_va, (oof_lgb[va_idx] + oof_xgb[va_idx] + oof_cb[va_idx]) / 3)
    print(f'  Fold {fold+1} GBM mean OOF mAP: {fold_map:.4f}\n')

print('GBM training complete.')

=== Training GBM Ensemble ===

--- GBM Fold 1/5 ---


  SMOTE: 2080 -> 10818 samples
Training until validation scores don't improve for 80 rounds


Early stopping, best iteration is:
[109]	valid_0's multi_logloss: 0.602177


0:	learn: 0.6616305	test: 0.3912357	best: 0.3912357 (0)	total: 118ms	remaining: 3m 56s


400:	learn: 0.9877384	test: 0.6842719	best: 0.6889031 (326)	total: 10.1s	remaining: 40.4s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.6889030558
bestIteration = 326

Shrink model to first 327 iterations.


...

Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.6937290896
bestIteration = 489

Shrink model to first 490 iterations.
  Fold 4 GBM mean OOF mAP: 0.7287

--- GBM Fold 5/5 ---
  SMOTE: 2081 -> 10827 samples
Training until validation scores don't improve for 80 rounds


Early stopping, best iteration is:
[132]	valid_0's multi_logloss: 0.594641


0:	learn: 0.6402612	test: 0.4157011	best: 0.4157011 (0)	total: 24.1ms	remaining: 48.1s


400:	learn: 0.9874660	test: 0.6883352	best: 0.6954655 (386)	total: 9.21s	remaining: 36.7s


Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.6988859884
bestIteration = 480

Shrink model to first 481 iterations.
  Fold 5 GBM mean OOF mAP: 0.7128

GBM training complete.


In [7]:
# Cell 7 — Optimise 4-Model Ensemble Weights & Evaluate
print('=== Optimising ensemble weights ===\n')

def blended(weights):
    w = np.abs(weights); w /= w.sum()
    pred = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cb + w[3]*oof_cnn
    mAP, _ = compute_map(y, pred)
    return -mAP

res   = minimize(blended, [0.25, 0.25, 0.25, 0.25], method='Nelder-Mead',
                 options={'maxiter': 2000, 'xatol': 1e-5})
w_opt = np.abs(res.x); w_opt /= w_opt.sum()
print(f'Optimal weights:')
print(f'  LGB={w_opt[0]:.3f}  XGB={w_opt[1]:.3f}  CB={w_opt[2]:.3f}  CNN={w_opt[3]:.3f}')

oof_blend  = w_opt[0]*oof_lgb  + w_opt[1]*oof_xgb  + w_opt[2]*oof_cb  + w_opt[3]*oof_cnn
test_blend = w_opt[0]*test_lgb + w_opt[1]*test_xgb + w_opt[2]*test_cb + w_opt[3]*test_cnn

cv_mAP, per_class = compute_map(y, oof_blend)
print_results(cv_mAP, per_class, label='E05 — 1D-CNN + GBM Ensemble (OOF)')

=== Optimising ensemble weights ===



Optimal weights:
  LGB=0.524  XGB=0.348  CB=0.119  CNN=0.009

  E05 — 1D-CNN + GBM Ensemble (OOF)

  Overall mAP: 0.7152

  Clutter        : 0.5709 <-- weak
  Cormorants     : 0.9381
  Pigeons        : 0.2128 <-- weak
  Ducks          : 0.6455
  Geese          : 0.7857
  Gulls          : 0.9552
  Birds of Prey  : 0.8854
  Waders         : 0.8265
  Songbirds      : 0.6166


In [8]:
# Cell 8 — Save Submission
save_submission(test_blend, name='e05_1dcnn_ensemble', cv_map=cv_mAP)
print('\nDone! Remember to log this run in EXPERIMENTS.md:')
print(f'| E05 | 2026-02-13 | e05_1dcnn_ensemble | {cv_mAP:.4f} | ... |')

Saved: e05_1dcnn_ensemble_0.7152_20260213_1812.csv (1872 rows)

Done! Remember to log this run in EXPERIMENTS.md:
| E05 | 2026-02-13 | e05_1dcnn_ensemble | 0.7152 | ... |
